# Outcome omnibus — Cochran's Q

Is there *any* difference in solve rate across backends? Omnibus test on the paired binary outcome matrix, plus the within-/across-tier breakdown.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
from collections import defaultdict

import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_outcome_matrix

summaries = aggregate_runs(records)
binary_ids, matrix = build_outcome_matrix(summaries, backends=backends, strategy=STRATEGY)
q = st.cochrans_q(matrix)
col_success = [sum(row[j] for row in matrix) for j in range(len(backends))]
print("Cochran's Q:", q)
print("solved per backend:", dict(zip(backends, col_success, strict=True)))

In [ ]:
fig, ax = plt.subplots()
ax.bar(backends, col_success, color="steelblue")
ax.set_ylabel(f"binaries solved (of {len(binary_ids)})")
ax.set_title(f"Outcome success by backend ({STRATEGY})")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
load_runs.save_figure(fig, "01_outcome_omnibus", "success_by_backend")
fig

## Difficulty tiers — within-tier and across-tier

Per the Step-6 decision, report success **within each tier** (per backend) and the **across-tier** gradient (pooled).

In [ ]:
tiers = load_runs.tiers(records)
within = defaultdict(lambda: defaultdict(list))  # backend -> tier -> [cell success]
for s in summaries:
    if s.strategy != STRATEGY:
        continue
    tier = load_runs.difficulty_tier(s.binary_id)
    if tier is not None:
        within[s.backend_name][tier].append(s.success)


def rate(values):
    return float(np.mean(values)) if values else 0.0


within_rate = {b: {t: rate(within[b][t]) for t in tiers} for b in backends}
across_rate = {t: rate([v for b in backends for v in within[b][t]]) for t in tiers}
print("across-tier success rate:", across_rate)

In [ ]:
fig, ax = plt.subplots()
width = 0.8 / max(1, len(backends))
x = np.arange(len(tiers))
for i, b in enumerate(backends):
    ax.bar(x + i * width, [within_rate[b][t] for t in tiers], width, label=b)
ax.plot(
    x + width * (len(backends) - 1) / 2,
    [across_rate[t] for t in tiers],
    "k--o",
    label="all backends",
)
ax.set_xticks(x + width * (len(backends) - 1) / 2)
ax.set_xticklabels([f"tier {t}" for t in tiers])
ax.set_ylabel("success rate")
ax.set_title(f"Success by difficulty tier ({STRATEGY})")
ax.legend(fontsize=8)
load_runs.save_figure(fig, "01_outcome_omnibus", "success_rate_by_tier")
fig

In [ ]:
from IPython.display import Markdown

verdict = "reject H0" if q.p_value < 0.05 else "fail to reject H0"
Markdown(f"""### Summary — outcome omnibus (Cochran's Q)

- Strategy held fixed: **{STRATEGY}**; binaries n = **{q.n}**, backends k = **{q.k}**.
- Q = **{q.statistic:.3f}**, df = {q.df}, p = **{q.p_value:.4g}** -> **{verdict}** at alpha = 0.05.
- Across-tier success gradient: {{ {", ".join(f"tier {t}: {across_rate[t]:.2f}" for t in tiers)} }}.

Figures: `analysis/figures/01_outcome_omnibus/`.""")